# Argos · Notebook 01 — Coleta e preparo dos dados

**Problema:** o pequeno produtor de milho planta sem saber se o ano vai ser bom ou ruim. Uma quebra de safra pode comprometer a renda da família inteira.

**Como a IA ajuda:** treinamos dois modelos com dados públicos de clima e de produção:
1. uma **regressão**, que estima quantos kg por hectare o produtor deve colher;
2. uma **classificação**, que diz se o ano tem risco de quebra de safra (baixo, médio ou alto).

Este notebook junta os dados e monta o arquivo que os outros notebooks usam (`dados/dataset_argos.csv`).

| Etapa | Notebook |
|---|---|
| Coleta e montagem do dataset | 01_coleta_e_preparo (este) |
| Análise exploratória (EDA) | 02_eda |
| Preparação, treino e avaliação da regressão | 03_regressao |
| Preparação, treino e avaliação da classificação | 04_classificacao |
| Exportação dos modelos (.joblib) | final do 03 e do 04 |
| API REST em Flask | api/api.py |

**Fontes:** IBGE (SIDRA, Produção Agrícola Municipal), Open-Meteo (clima histórico) e NOAA (índice do El Niño). O endereço e a data de cada download ficam registrados em `dados/log_downloads.txt`.

## 0. Configuração
Importamos as bibliotecas e definimos as constantes usadas no notebook: os anos (2005 a 2024), os estados candidatos, os 8 municípios que já estão no banco de dados do Argos e as mesorregiões que vamos excluir (explicado na seção 2). A função `registrar_download` anota cada endereço baixado num arquivo de texto, para o README citar as fontes com data.

In [1]:
import sys, json, time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import sklearn

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| requests", requests.__version__)
assert Path.cwd().name == "notebooks", f"Rode o notebook de dentro de notebooks/ (cwd = {Path.cwd()})"

SEED = 2279
ANOS = list(range(2005, 2025))
PASTA_DADOS = Path("../dados")
PASTA_CLIMA = PASTA_DADOS / "clima_bruto"
PASTA_CLIMA.mkdir(parents=True, exist_ok=True)

UF_CODIGO = {"PE": 26, "BA": 29, "RN": 24, "PI": 22, "RS": 43, "SC": 42, "PR": 41}
UF_NOME = {"PE": "Pernambuco", "BA": "Bahia", "RN": "Rio Grande do Norte", "PI": "Piauí",
           "RS": "Rio Grande do Sul", "SC": "Santa Catarina", "PR": "Paraná"}
UF_REGIAO = {"PE": "NORDESTE", "BA": "NORDESTE", "RN": "NORDESTE", "PI": "NORDESTE",
             "RS": "SUL", "SC": "SUL", "PR": "SUL"}
MUNICIPIOS_BANCO = [2611101, 2918407, 2408003, 2208007, 4302105, 4314100, 4204202, 4104808]
MESO_EXCLUIDAS = ["Extremo Oeste Baiano", "Sudoeste Piauiense"]   # cerrado do MATOPIBA
POR_REGIAO = 15

def registrar_download(fonte, url):
    with open(PASTA_DADOS / "log_downloads.txt", "a", encoding="utf-8") as arq:
        arq.write(f"{datetime.now():%d/%m/%Y %H:%M} | {fonte} | {url}\n")

Python 3.12.10 | pandas 3.1.1 | numpy 2.5.5 | scikit-learn 1.9.1 | requests 2.34.2


## 1. Rendimento do milho (IBGE/SIDRA)
Usamos a tabela 1612 do SIDRA (Produção Agrícola Municipal, lavouras temporárias), com:
- variável 112: rendimento médio, em kg/ha;
- variável 216: área colhida, em hectares;
- produto 2711: milho em grão;
- todos os municípios de PE, BA, RN e PI (Nordeste) e de RS, SC e PR (Sul), de 2005 a 2024.

Baixamos pela API do SIDRA, um estado por vez, e salvamos o resultado em `dados/sidra_milho.csv`. Nas próximas execuções, o notebook lê o arquivo salvo e não baixa de novo.

Sobre os símbolos do IBGE: "-" significa zero, "..." significa dado não disponível e "X" é dado omitido por sigilo. No rendimento, o "-" aparece quando não houve área colhida, e aí não dá para calcular kg/ha (seria uma divisão por zero). Por isso tratamos os três símbolos como **ausente**, e não como zero.

In [2]:
ARQ_SIDRA = PASTA_DADOS / "sidra_milho.csv"

if ARQ_SIDRA.exists():
    sidra = pd.read_csv(ARQ_SIDRA, dtype=str, encoding="utf-8")
    print("Lido do arquivo local:", ARQ_SIDRA)
else:
    periodos = ",".join(str(ano) for ano in ANOS)
    partes = []
    for uf, cod_uf in UF_CODIGO.items():
        url = (f"https://apisidra.ibge.gov.br/values/t/1612/n6/in%20n3%20{cod_uf}"
               f"/v/112,216/p/{periodos}/c81/2711")
        resposta = requests.get(url, timeout=180)
        resposta.raise_for_status()
        linhas = resposta.json()
        cabecalho = linhas[0]                    # a 1ª linha da API traz o nome das colunas
        assert cabecalho["D1N"] == "Município" and cabecalho["D3N"] == "Ano", cabecalho
        parte = pd.DataFrame(linhas[1:])
        parte["sg_uf"] = uf
        partes.append(parte)
        registrar_download("SIDRA/IBGE - PAM tabela 1612", url)
        print(uf, len(parte), "linhas")
        time.sleep(1)
    sidra = pd.concat(partes, ignore_index=True)
    sidra = sidra.rename(columns={"D1C": "cd_municipio", "D1N": "nm_municipio", "D2C": "cd_variavel",
                                  "D2N": "variavel", "D3C": "ano", "D4N": "produto", "V": "valor"})
    sidra = sidra[["cd_municipio", "nm_municipio", "sg_uf", "ano", "cd_variavel", "variavel", "produto", "valor"]]
    sidra.to_csv(ARQ_SIDRA, index=False, encoding="utf-8")

print(sidra.shape)
print(sidra.groupby(["cd_variavel", "variavel"]).size())
print("Produto:", sidra["produto"].unique())
print("Símbolos encontrados:")
print(sidra.loc[~sidra["valor"].str.isdigit(), "valor"].value_counts())

Lido do arquivo local: ..\dados\sidra_milho.csv
(87360, 8)
cd_variavel  variavel                    
112          Rendimento médio da produção    43680
216          Área colhida                    43680
dtype: int64
Produto: <StringArray>
['Milho (em grão)']
Length: 1, dtype: str
Símbolos encontrados:
valor
-      6490
...     566
Name: count, dtype: int64


A tabela do SIDRA vem com uma linha para cada variável. Aqui convertemos os valores para número (os símbolos viram `NaN`, que é o "ausente" do pandas) e reorganizamos com `pivot`, para ficar com uma linha por município e ano, com o rendimento e a área colhida lado a lado.

In [3]:
num = sidra.copy()
num["valor"] = pd.to_numeric(num["valor"], errors="coerce")   # "...", "-", "X" -> NaN
num["cd_municipio"] = num["cd_municipio"].astype(int)
num["ano"] = num["ano"].astype(int)
num["nm_municipio"] = num["nm_municipio"].str.rsplit(" - ", n=1).str[0]   # "Petrolina - PE" -> "Petrolina"

larga = num.pivot(index=["cd_municipio", "nm_municipio", "sg_uf", "ano"],
                  columns="cd_variavel", values="valor").reset_index()
larga = larga.rename(columns={"112": "rend_kg_ha", "216": "area_colhida_ha"})
larga.columns.name = None
print(larga.shape)
larga.head()

(43680, 6)


,cd_municipio,nm_municipio,sg_uf,ano,rend_kg_ha,area_colhida_ha
0,2200053,Acauã,PI,2005,240.0,3450.0
1,2200053,Acauã,PI,2006,503.0,3435.0
2,2200053,Acauã,PI,2007,59.0,3441.0
3,2200053,Acauã,PI,2008,360.0,3600.0
4,2200053,Acauã,PI,2009,749.0,3650.0


Buscamos também a mesorregião de cada município, na API de Localidades do IBGE. Ela é usada na regra 3 do critério de escolha (seção 2).

In [4]:
ARQ_MESO = PASTA_DADOS / "ibge_mesorregioes.csv"

if ARQ_MESO.exists():
    meso = pd.read_csv(ARQ_MESO, encoding="utf-8")
else:
    registros = []
    for uf, cod_uf in UF_CODIGO.items():
        url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{cod_uf}/municipios"
        resposta = requests.get(url, timeout=60)
        resposta.raise_for_status()
        for m in resposta.json():
            micro = m.get("microrregiao") or {}
            registros.append({"cd_municipio": m["id"],
                              "nm_mesorregiao": (micro.get("mesorregiao") or {}).get("nome")})
        registrar_download("IBGE - API de Localidades", url)
    meso = pd.DataFrame(registros)
    meso.to_csv(ARQ_MESO, index=False, encoding="utf-8")

print("Sem mesorregião:", meso["nm_mesorregiao"].isna().sum())
print(meso.loc[meso["nm_mesorregiao"].isin(MESO_EXCLUIDAS), "nm_mesorregiao"].value_counts())

Sem mesorregião: 0
nm_mesorregiao
Sudoeste Piauiense      62
Extremo Oeste Baiano    24
Name: count, dtype: int64


## 2. Como escolhemos os 30 municípios
A escolha seguiu regras fixas:
1. **Candidatos:** todos os municípios de PE, BA, RN e PI (Nordeste) e de RS, SC e PR (Sul).
2. **Série completa:** só entra quem tem rendimento de milho nos 20 anos (2005 a 2024). Se faltar um ano, o município sai, porque um buraco na série estraga a média dos 5 anos anteriores (seção 6).
3. **Fora do MATOPIBA:** tiramos as mesorregiões Extremo Oeste Baiano e Sudoeste Piauiense. Elas ficam no cerrado, com lavouras grandes e mecanizadas, e não representam o pequeno produtor do semiárido que o Argos quer atender.
4. **Municípios do banco primeiro:** os 8 municípios que já usamos no banco de dados do Argos entram na frente, desde que passem nas regras 2 e 3.
5. **Desempate:** as vagas que sobram (até 15 por região) ficam com os municípios de maior área colhida média, porque mais área colhida deixa a série mais confiável.

In [5]:
resumo = (larga.groupby(["cd_municipio", "nm_municipio", "sg_uf"], as_index=False)
               .agg(anos_com_rend=("rend_kg_ha", "count"),
                    area_media_ha=("area_colhida_ha", "mean")))
resumo["ds_regiao"] = resumo["sg_uf"].map(UF_REGIAO)
resumo = resumo.merge(meso, on="cd_municipio", how="left")

completo = resumo["anos_com_rend"] == len(ANOS)
fora_matopiba = ~resumo["nm_mesorregiao"].isin(MESO_EXCLUIDAS)
candidatos = resumo[completo & fora_matopiba]

print("Baixados:", len(resumo), "| série completa:", completo.sum(), "| candidatos finais:", len(candidatos))
print(candidatos["ds_regiao"].value_counts())
print("\nOs 8 do banco:")
print(resumo[resumo["cd_municipio"].isin(MUNICIPIOS_BANCO)]
      [["cd_municipio", "nm_municipio", "sg_uf", "anos_com_rend", "area_media_ha"]])

escolhidos = []
for regiao in ["NORDESTE", "SUL"]:
    cand = candidatos[candidatos["ds_regiao"] == regiao]
    do_banco = cand[cand["cd_municipio"].isin(MUNICIPIOS_BANCO)]
    outros = cand[~cand["cd_municipio"].isin(MUNICIPIOS_BANCO)].sort_values("area_media_ha", ascending=False)
    escolhidos.append(pd.concat([do_banco, outros.head(POR_REGIAO - len(do_banco))]))

municipios = pd.concat(escolhidos, ignore_index=True)
print(municipios["ds_regiao"].value_counts())
municipios[["cd_municipio", "nm_municipio", "sg_uf", "ds_regiao", "area_media_ha", "nm_mesorregiao"]]

Baixados: 2184 | série completa: 1461 | candidatos finais: 1390
ds_regiao
SUL         1114
NORDESTE     276
Name: count, dtype: int64

Os 8 do banco:
      cd_municipio     nm_municipio sg_uf  anos_com_rend  area_media_ha
159        2208007            Picos    PI             19    2661.789474
311        2408003          Mossoró    RN             19    1905.421053
516        2611101        Petrolina    PE             17    1934.117647
793        2918407         Juazeiro    BA             18     806.611111
1061       4104808         Cascavel    PR             20   42458.350000
1458       4204202          Chapecó    SC             20    7380.500000
1729       4302105  Bento Gonçalves    RS             20     758.450000
1991       4314100      Passo Fundo    RS             20    1855.000000
ds_regiao
NORDESTE    15
SUL         15
Name: count, dtype: int64


,cd_municipio,nm_municipio,sg_uf,ds_regiao,area_media_ha,nm_mesorregiao
0,2900355,Adustina,BA,NORDESTE,26256.50,Nordeste Baiano
1,2923803,Paripiranga,BA,NORDESTE,25960.00,Nordeste Baiano
2,2910750,Fátima,BA,NORDESTE,9680.00,Nordeste Baiano
3,2930766,Sítio do Quinto,BA,NORDESTE,8303.85,Nordeste Baiano
4,2910800,Feira de Santana,BA,NORDESTE,7449.85,Centro Norte Baiano
5,2907806,Cícero Dantas,BA,NORDESTE,6632.75,Nordeste Baiano
6,2916500,Itapicuru,BA,NORDESTE,6421.00,Nordeste Baiano
7,2913705,Inhambupe,BA,NORDESTE,4969.00,Nordeste Baiano
8,2913101,Ibititá,BA,NORDESTE,4404.55,Centro Norte Baiano
9,2922250,Muquém do São Francisco,BA,NORDESTE,4160.80,Vale São-Franciscano da Bahia


**Resultado da escolha.** Os 4 municípios do banco que são do Nordeste ficaram de fora pela regra 2: Petrolina (sem dado em 2012, 2013 e 2014), Juazeiro (2005 e 2019), Mossoró (2017) e Picos (2013). Nesses anos o IBGE registra "-" na área colhida, ou seja, não houve colheita de milho. Isso cria um viés: os anos de perda total do semiárido ficam fora da amostra, e o modelo tende a subestimar o risco nos piores anos. Por esse motivo, na demonstração da API usamos outro município do semiárido no lugar de Petrolina (Araci/BA, ver notebook 02).

A amostra também ficou concentrada em dois estados: no Nordeste, 13 dos 15 municípios são da Bahia (quase todos do Nordeste Baiano, que é semiárido); no Sul, 12 dos 15 são do Paraná. Isso é consequência do critério e entra nas limitações do projeto.

Para baixar o clima, precisamos da latitude e da longitude de cada município. Usamos a busca de localidades da própria Open-Meteo e conferimos se o resultado é do estado certo, já que existem municípios com o mesmo nome em estados diferentes. No fim, verificamos se alguma coordenada caiu fora da faixa esperada para a região.

In [6]:
ARQ_MUNICIPIOS = PASTA_DADOS / "municipios.csv"

def buscar_coordenadas(nome, uf):
    params = {"name": nome, "count": 10, "language": "pt", "countryCode": "BR"}
    resposta = requests.get("https://geocoding-api.open-meteo.com/v1/search", params=params, timeout=30)
    resposta.raise_for_status()
    for lugar in resposta.json().get("results", []):
        if lugar.get("admin1") == UF_NOME[uf]:          # mesmo nome em outro estado é descartado
            return lugar["latitude"], lugar["longitude"]
    return np.nan, np.nan

if ARQ_MUNICIPIOS.exists():
    municipios = pd.read_csv(ARQ_MUNICIPIOS, encoding="utf-8")
    print("Lido do arquivo local")
else:
    latitudes, longitudes = [], []
    for linha in municipios.itertuples():
        lat, lon = buscar_coordenadas(linha.nm_municipio, linha.sg_uf)
        latitudes.append(lat)
        longitudes.append(lon)
        time.sleep(0.5)
    municipios["latitude"] = latitudes
    municipios["longitude"] = longitudes
    registrar_download("Open-Meteo Geocoding", "https://geocoding-api.open-meteo.com/v1/search")
    municipios = municipios[["cd_municipio", "nm_municipio", "sg_uf", "ds_regiao",
                             "latitude", "longitude", "area_media_ha", "nm_mesorregiao"]]
    municipios.to_csv(ARQ_MUNICIPIOS, index=False, encoding="utf-8")

suspeitos = municipios[((municipios["ds_regiao"] == "NORDESTE") & ~municipios["latitude"].between(-16, -2)) |
                       ((municipios["ds_regiao"] == "SUL") & ~municipios["latitude"].between(-34, -22))]
print("Sem coordenada ou fora da faixa:", len(suspeitos))
municipios

Lido do arquivo local
Sem coordenada ou fora da faixa: 0


,cd_municipio,nm_municipio,sg_uf,ds_regiao,latitude,longitude,area_media_ha,nm_mesorregiao
0,2900355,Adustina,BA,NORDESTE,-10.53333,-38.10000,26256.50,Nordeste Baiano
1,2923803,Paripiranga,BA,NORDESTE,-10.68750,-37.86167,25960.00,Nordeste Baiano
2,2910750,Fátima,BA,NORDESTE,-10.60000,-38.21667,9680.00,Nordeste Baiano
3,2930766,Sítio do Quinto,BA,NORDESTE,-10.36565,-38.20280,8303.85,Nordeste Baiano
4,2910800,Feira de Santana,BA,NORDESTE,-12.26667,-38.96667,7449.85,Centro Norte Baiano
5,2907806,Cícero Dantas,BA,NORDESTE,-10.60000,-38.38333,6632.75,Nordeste Baiano
6,2916500,Itapicuru,BA,NORDESTE,-11.31667,-38.23333,6421.00,Nordeste Baiano
7,2913705,Inhambupe,BA,NORDESTE,-11.78444,-38.35306,4969.00,Nordeste Baiano
8,2913101,Ibititá,BA,NORDESTE,-11.54722,-41.97750,4404.55,Centro Norte Baiano
9,2922250,Muquém do São Francisco,BA,NORDESTE,-12.06597,-43.54934,4160.80,Vale São-Franciscano da Bahia


## 3. Clima diário (Open-Meteo)
Para cada município, fazemos uma chamada à API de histórico da Open-Meteo pedindo a chuva diária e as temperaturas média e máxima de 2005 a 2024. Esse dado vem de reanálise (um modelo do clima corrigido com observações) para o ponto mais próximo da sede do município. Não é de uma estação meteorológica, mas tem a vantagem de não ter dias faltando.

Cada resposta é salva em `dados/clima_bruto/<código do município>.json`. Assim, o notebook pode rodar de novo sem internet e sem gastar o limite de uso gratuito da API. Primeiro testamos com dois municípios, um de cada região, para conferir se os números fazem sentido; depois baixamos os outros, com uma pausa entre as chamadas.

In [7]:
URL_CLIMA = "https://archive-api.open-meteo.com/v1/archive"

def baixar_clima(cd_municipio, lat, lon):
    arquivo = PASTA_CLIMA / f"{cd_municipio}.json"
    if arquivo.exists():
        return json.loads(arquivo.read_text(encoding="utf-8"))
    params = {"latitude": lat, "longitude": lon, "start_date": "2005-01-01", "end_date": "2024-12-31",
              "daily": "precipitation_sum,temperature_2m_mean,temperature_2m_max",
              "timezone": "America/Sao_Paulo"}
    resposta = requests.get(URL_CLIMA, params=params, timeout=180)
    if resposta.status_code == 429:
        raise RuntimeError("Limite da Open-Meteo atingido; rode de novo mais tarde. " + resposta.text)
    resposta.raise_for_status()
    dados = resposta.json()
    arquivo.write_text(json.dumps(dados), encoding="utf-8")
    registrar_download("Open-Meteo Historical (archive-api)", resposta.url)
    return dados

def clima_diario(dados):
    diario = pd.DataFrame(dados["daily"])
    diario["time"] = pd.to_datetime(diario["time"])
    return diario

teste = municipios[municipios["cd_municipio"].isin([2611101, 4314100])]   # Petrolina e Passo Fundo
if len(teste) < 2:
    teste = municipios.groupby("ds_regiao").head(1)

for linha in teste.itertuples():
    dados = baixar_clima(linha.cd_municipio, linha.latitude, linha.longitude)
    diario = clima_diario(dados)
    print(f"\n{linha.nm_municipio}/{linha.sg_uf} | grade: {dados['latitude']}, {dados['longitude']} "
          f"| altitude {dados['elevation']} m")
    print("Dias:", len(diario), "|", diario["time"].min().date(), "a", diario["time"].max().date())
    print("Ausentes:", diario.isna().sum().to_dict())
    anual = diario.groupby(diario["time"].dt.year).agg(chuva_mm=("precipitation_sum", "sum"),
                                                        temp_media=("temperature_2m_mean", "mean"),
                                                        temp_max_media=("temperature_2m_max", "mean"))
    print(anual.round(1).head(3))
    print("Chuva anual média:", round(anual["chuva_mm"].mean()), "mm")


Adustina/BA | grade: -10.5096655, -38.061676 | altitude 285.0 m
Dias: 7305 | 2005-01-01 a 2024-12-31
Ausentes: {'time': 0, 'precipitation_sum': 0, 'temperature_2m_mean': 0, 'temperature_2m_max': 0}
      chuva_mm  temp_media  temp_max_media
time                                      
2005     398.3        24.8            30.3
2006     451.9        24.6            29.8
2007     457.8        24.2            29.4
Chuva anual média: 467 mm

Cascavel/PR | grade: -24.991213, -53.476868 | altitude 781.0 m
Dias: 7305 | 2005-01-01 a 2024-12-31
Ausentes: {'time': 0, 'precipitation_sum': 0, 'temperature_2m_mean': 0, 'temperature_2m_max': 0}
      chuva_mm  temp_media  temp_max_media
time                                      
2005    1471.8        19.6            24.3
2006    1463.7        19.6            24.3
2007    1349.7        19.7            24.5
Chuva anual média: 1646 mm


In [8]:
falhas = []
for linha in municipios.itertuples():
    if (PASTA_CLIMA / f"{linha.cd_municipio}.json").exists():
        continue
    try:
        baixar_clima(linha.cd_municipio, linha.latitude, linha.longitude)
        print("ok", linha.nm_municipio)
        time.sleep(25)                      # respeita o limite por minuto/hora
    except Exception as erro:
        print("FALHOU", linha.nm_municipio, "->", erro)
        falhas.append(linha.nm_municipio)
        break                               # para no 1º erro; rodar de novo continua daqui
print("Arquivos de clima:", len(list(PASTA_CLIMA.glob("*.json"))), "de", len(municipios), "| falhas:", falhas)

Arquivos de clima: 30 de 30 | falhas: []


## 4. Índice do El Niño (ONI, da NOAA)
O ONI mede o quanto a temperatura do Oceano Pacífico está acima ou abaixo do normal. Valores acima de +0,5 indicam El Niño, que costuma trazer seca ao Nordeste, e abaixo de −0,5 indicam La Niña, que costuma trazer seca ao Sul. A NOAA publica um valor para cada trimestre (DJF, JFM, ...), e nós usamos a média dos 12 trimestres de cada ano.

In [9]:
ARQ_ONI_BRUTO = PASTA_DADOS / "oni_bruto.txt"
ARQ_ONI = PASTA_DADOS / "oni.csv"
URL_ONI = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

if not ARQ_ONI_BRUTO.exists():
    resposta = requests.get(URL_ONI, timeout=60)
    resposta.raise_for_status()
    ARQ_ONI_BRUTO.write_text(resposta.text, encoding="utf-8")
    registrar_download("NOAA CPC - ONI", URL_ONI)

oni_trimestral = pd.read_csv(ARQ_ONI_BRUTO, sep=r"\s+")
print(oni_trimestral.head(3))

oni = (oni_trimestral[oni_trimestral["YR"].isin(ANOS)]
       .groupby("YR", as_index=False)["ANOM"].mean()
       .rename(columns={"YR": "ano", "ANOM": "oni"}))
oni["oni"] = oni["oni"].round(2)
assert len(oni) == len(ANOS), oni
oni.to_csv(ARQ_ONI, index=False, encoding="utf-8")
oni.set_index("ano").T

  SEAS    YR  TOTAL  ANOM
0  DJF  1950  25.01 -1.32
1  JFM  1950  25.36 -1.20
2  FMA  1950  25.88 -1.12


ano,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
oni,0.03,0.04,-0.61,-0.81,0.26,-0.43,-0.77,-0.07,-0.26,0.23,1.48,0.49,-0.06,0.14,0.65,-0.21,-0.65,-0.78,0.81,0.41


## 5. Clima por município e ano
Transformamos os dados diários em um resumo por ano (de janeiro a dezembro):

| Coluna | Como calculamos |
|---|---|
| chuva_total_mm | soma da chuva de todos os dias do ano |
| temp_media | média da temperatura média diária |
| temp_max_media | média da temperatura máxima diária |
| dias_acima_35 | quantidade de dias com máxima acima de 35 °C |
| maior_seq_seca | maior sequência de dias seguidos com menos de 1 mm de chuva |

Uma limitação: a sequência seca é contada dentro de cada ano, então uma estiagem que começa em dezembro e continua em janeiro é contada como duas.

In [10]:
def maior_sequencia_seca(chuva_diaria):
    maior = 0
    atual = 0
    for mm in chuva_diaria:
        if mm < 1:
            atual += 1
            if atual > maior:
                maior = atual
        else:
            atual = 0
    return maior

partes = []
for linha in municipios.itertuples():
    dados = json.loads((PASTA_CLIMA / f"{linha.cd_municipio}.json").read_text(encoding="utf-8"))
    diario = clima_diario(dados)
    diario["ano"] = diario["time"].dt.year
    anual = diario.groupby("ano", as_index=False).agg(
        chuva_total_mm=("precipitation_sum", "sum"),
        temp_media=("temperature_2m_mean", "mean"),
        temp_max_media=("temperature_2m_max", "mean"),
        dias_acima_35=("temperature_2m_max", lambda t: int((t > 35).sum())),
        maior_seq_seca=("precipitation_sum", maior_sequencia_seca),
    )
    anual["cd_municipio"] = linha.cd_municipio
    partes.append(anual)

clima_anual = pd.concat(partes, ignore_index=True)
print(clima_anual.shape)
clima_anual.describe().round(1)

(600, 7)


,ano,chuva_total_mm,temp_media,temp_max_media,dias_acima_35,maior_seq_seca,cd_municipio
count,600.0,600.0,600.0,600.0,600.0,600.0,600.0
mean,2014.5,1109.4,23.0,28.2,14.4,37.7,3508545.8
std,5.8,588.4,2.6,2.8,24.1,39.3,654721.0
min,2005.0,166.6,15.9,20.8,0.0,7.0,2210409.0
25%,2009.8,570.7,21.6,26.6,0.0,18.0,2913101.0
50%,2014.5,1031.8,23.2,28.5,5.0,24.0,3516383.0
75%,2019.2,1615.5,24.8,30.1,17.0,34.0,4125704.0
max,2024.0,2653.8,29.1,34.4,153.0,215.0,4314100.0


## 6. Média histórica e alvo
- **media_5a:** média do rendimento dos 5 anos **anteriores** ao ano da linha. Usamos `shift(1)` antes do `rolling(5)` para que o próprio ano não entre na média. Se entrasse, a variável já carregaria parte da resposta que o modelo precisa prever (vazamento de dados). Como 2005 a 2009 não têm 5 anos anteriores, essas linhas saem, e o dataset começa em 2010.
- **quebra:** vale 1 quando o rendimento do ano fica abaixo de 80% da `media_5a`, ou seja, quando o município colheu mais de 20% a menos que o seu normal recente. Essa regra foi definida pelo grupo. Comparamos cada município com ele mesmo porque o Sul colhe muito mais por hectare que o semiárido; com um valor fixo de kg/ha, quase todo ano do Nordeste seria "quebra". O corte de 80% foi definido antes de treinar os modelos e não foi mudado depois.

In [11]:
rend = larga[larga["cd_municipio"].isin(municipios["cd_municipio"])][
    ["cd_municipio", "nm_municipio", "sg_uf", "ano", "rend_kg_ha"]]

df = rend.merge(clima_anual, on=["cd_municipio", "ano"], how="inner")
df = df.merge(oni, on="ano", how="left")
df["regiao_sul"] = (df["sg_uf"].map(UF_REGIAO) == "SUL").astype(int)

df = df.sort_values(["cd_municipio", "ano"]).reset_index(drop=True)
df["media_5a"] = df.groupby("cd_municipio")["rend_kg_ha"].transform(
    lambda s: s.shift(1).rolling(5).mean()).round(1)
print("Antes de tirar 2005-2009:", df.shape)
df = df.dropna(subset=["media_5a"]).reset_index(drop=True)
df["quebra"] = (df["rend_kg_ha"] < 0.8 * df["media_5a"]).astype(int)

COLUNAS = ["cd_municipio", "nm_municipio", "sg_uf", "ano",
           "chuva_total_mm", "temp_media", "temp_max_media", "dias_acima_35",
           "maior_seq_seca", "oni", "regiao_sul", "media_5a", "rend_kg_ha", "quebra"]
df = df[COLUNAS].round({"chuva_total_mm": 1, "temp_media": 2, "temp_max_media": 2})
df.head()

Antes de tirar 2005-2009: (600, 13)


,cd_municipio,nm_municipio,sg_uf,ano,chuva_total_mm,temp_media,temp_max_media,dias_acima_35,maior_seq_seca,oni,regiao_sul,media_5a,rend_kg_ha,quebra
0,2210409,São Miguel do Tapuio,PI,2010,612.4,28.70,33.83,121,124,-0.43,0,256.0,40.0,1
1,2210409,São Miguel do Tapuio,PI,2011,1430.4,27.02,31.81,42,74,-0.77,0,216.0,320.0,0
2,2210409,São Miguel do Tapuio,PI,2012,294.0,28.90,34.24,143,161,-0.07,0,152.0,160.0,0
3,2210409,São Miguel do Tapuio,PI,2013,570.5,28.56,33.73,124,90,-0.26,0,176.0,40.0,1
4,2210409,São Miguel do Tapuio,PI,2014,643.9,28.11,33.23,88,96,0.23,0,144.2,118.0,0


Antes de salvar, conferimos o dataset: se tem os 8 campos combinados com o front, se não tem valores ausentes, se todo ano tem os 30 municípios e se a `media_5a` realmente não inclui o próprio ano.

In [12]:
ENTRADAS_CONTRATO = ["chuva_total_mm", "temp_media", "temp_max_media", "dias_acima_35",
                     "maior_seq_seca", "oni", "regiao_sul", "media_5a"]
assert all(c in df.columns for c in ENTRADAS_CONTRATO)

print("Formato:", df.shape, "| ausentes:", df.isna().sum().sum())
print("Municípios por ano:", df.groupby("ano").size().unique(), "| anos:", df["ano"].min(), "a", df["ano"].max())

exemplo = df["cd_municipio"].iloc[0]
serie = rend[rend["cd_municipio"] == exemplo].set_index("ano")["rend_kg_ha"]
media_2010 = df.loc[(df["cd_municipio"] == exemplo) & (df["ano"] == 2010), "media_5a"].item()
print("Prova do shift(1): média 2005-2009 =", round(serie.loc[2005:2009].mean(), 1), "| media_5a de 2010 =", media_2010)

periodo = np.where(df["ano"] <= 2019, "treino 2010-2019", "teste 2020-2024")
regiao = df["sg_uf"].map(UF_REGIAO)
print(f"\nQuebra no total: {df['quebra'].mean():.1%}")
print(df.assign(periodo=periodo, regiao=regiao)
        .groupby(["periodo", "regiao"])["quebra"].agg(quebras="sum", linhas="count", proporcao="mean").round(3))

ARQ_DATASET = PASTA_DADOS / "dataset_argos.csv"
df.to_csv(ARQ_DATASET, index=False, encoding="utf-8")
print("\nSalvo:", ARQ_DATASET, pd.read_csv(ARQ_DATASET).shape)

Formato: (450, 14) | ausentes: 0
Municípios por ano: [30] | anos: 2010 a 2024
Prova do shift(1): média 2005-2009 = 256.0 | media_5a de 2010 = 256.0

Quebra no total: 27.3%
                           quebras  linhas  proporcao
periodo          regiao                              
teste 2020-2024  NORDESTE       26      75      0.347
                 SUL            26      75      0.347
treino 2010-2019 NORDESTE       63     150      0.420
                 SUL             8     150      0.053

Salvo: ..\dados\dataset_argos.csv (450, 14)


**Interpretação.** O dataset final tem 450 linhas (30 municípios × 15 anos, de 2010 a 2024), 8 variáveis de entrada e 2 alvos, sem nenhum valor ausente. A quebra aparece em 27,3% das linhas: 39,6% no Nordeste (89 de 225) e 15,1% no Sul (34 de 225). No Sul, porém, ela quase não aparece nos anos de treino (8 de 150 linhas, 5,3%) e sobe para 34,7% nos anos de teste, por causa das secas de La Niña de 2020 a 2022. Ou seja, no Sul o modelo vai ser cobrado justamente no tipo de ano que ele quase não viu no treino. Vamos levar isso em conta ao interpretar as métricas.